# Enhancer–gene regulation from random walks + selective SSMs

## 1. Data
Raw data comes from `download_data.py` (sources listed in `data_sources.py`). Here we preprocess everything into three tables:
`crispr` (labeled pairs), `nodes` (candidate elements = graph nodes), and `hic` (contacts between node bins).

In [ ]:
import os
import time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool

import hicstraw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyfaidx
import pyranges as pr
import pysam

from data_sources import HIC_URL, data_directory, processed_dir, raw_dir, raw_path

In [ ]:
directory = data_directory()
RAW = raw_dir(directory)
PROC = processed_dir(directory)
PROC.mkdir(parents=True, exist_ok=True)

CHROMS = ["chr" + str(i) for i in range(1, 23)] + ["chrX"]

BAM_1 = raw_path(RAW, "atac_bam_1")
BAM_2 = raw_path(RAW, "atac_bam_2")
BAMS = [str(BAM_1), str(BAM_2)]

# ABC-style contact settings.
BIN_SIZE = 5_000
MAX_DIST = 5_000_000

In [ ]:
def PR(frame):
    """Convert chrom/start/end columns to a PyRanges object."""
    columns = {"chrom": "Chromosome", "start": "Start", "end": "End"}
    return pr.PyRanges(frame.rename(columns=columns))


def DF(ranges):
    """Convert a PyRanges object back to our pandas column names."""
    columns = {"Chromosome": "chrom", "Start": "start", "End": "end"}
    return ranges.df.rename(columns=columns)

### CRISPR benchmark

In [ ]:
# annotated genes in hg38
GENES_FILE = raw_path(RAW, "genes")
genes = pd.read_csv(GENES_FILE, sep="\t")
genes = genes.rename(columns={"#chr": "chrom", "name": "gene"})
genes = genes[genes["chrom"].isin(CHROMS)].copy()

# print(genes.head())

# make tss column
genes["tss"] = np.where(
    genes["strand"] == "+",
    genes["start"],
    genes["end"] - 1,
)

  chrom   start     end       gene  score strand       Ensembl_ID  \
0  chr1   34610   36081    FAM138A      0      -  ENSG00000237613   
1  chr1   34610   36081    FAM138F      0      -  ENSG00000282591   
2  chr1   69090   70008      OR4F5      0      +  ENSG00000186092   
3  chr1  817370  819834     FAM87B      0      +  ENSG00000177757   
4  chr1  826205  827522  LINC00115      0      -  ENSG00000225880   

        gene_type  
0         lincRNA  
1         lincRNA  
2  protein_coding  
3         lincRNA  
4         lincRNA  


,pairs,positives,rate
Dataset,,,
Gasperini2019,5299,360,0.067937
Nasser2021,3751,88,0.023460
Schraivogel2020,1306,23,0.017611


In [ ]:
# labeled enhancer-gene pairs
CRISPR_FILE = raw_path(RAW, "crispr")
crispr = pd.read_csv(CRISPR_FILE, sep="\t")

crispr = crispr.rename(columns={
    "chromStart": "start",
    "chromEnd": "end",
    "measuredGeneSymbol": "gene",
    "Regulated": "label",
})

# print(crispr[["gene", "label", "distanceToTSS"]].head())
# print(crispr.head())

# supply tss from `genes`
tss_by_gene = genes.set_index("gene")["tss"]
crispr["tss"] = crispr["gene"].map(tss_by_gene)
crispr["center"] = (crispr["start"] + crispr["end"]) // 2

crispr = crispr[[
    "chrom", "start", "end", "center", "gene", "tss",
    "distanceToTSS", "label", "Dataset",
]]

crispr_summary = crispr.groupby("Dataset")["label"].agg(
    pairs="size",
    positives="sum",
    rate="mean",
)
crispr_summary

,pairs,positives,rate
Dataset,,,
Gasperini2019,5299,360,0.067937
Nasser2021,3751,88,0.023460
Schraivogel2020,1306,23,0.017611


### Candidate elements (nodes)
ABC recipe: ATAC summit ±250 bp windows, merged, plus every promoter (TSS ±250 bp), minus blocklist. ENCODE's peak set has <150k distinct regions, so ABC's top-150k cut keeps everything.

In [ ]:
# use ABC's blocklist
BLOCKLIST_FILE = raw_path(RAW, "blocklist")
blocklist = pd.read_csv(
    BLOCKLIST_FILE,
    sep="\t",
    header=None,
    usecols=[0, 1, 2],
    names=["chrom", "start", "end"],
)
block = PR(blocklist)

# peak regions
PEAKS_FILE = raw_path(RAW, "atac_peaks")
peaks = pd.read_csv(
    PEAKS_FILE,
    sep="\t",
    header=None,
    usecols=[0, 1, 9],
    names=["chrom", "start", "summit"],
)
print(peaks.head())
peaks = peaks[peaks["chrom"].isin(CHROMS)].copy()
peaks["start"] = (peaks["start"] + peaks["summit"] - 250).clip(lower=0)
peaks["end"] = peaks["start"] + 500
peaks = peaks.drop_duplicates(["chrom", "start"])

# Add a 500 bp window centered on each annotated gene TSS.
promoters = genes[["chrom", "tss"]].copy()
promoters["start"] = promoters["tss"] - 250
promoters["end"] = promoters["tss"] + 250

# Merge overlapping candidates and remove blocked regions.
peak_regions = PR(peaks[["chrom", "start", "end"]])
promoter_regions = PR(promoters[["chrom", "start", "end"]])
combined_regions = pr.concat([peak_regions, promoter_regions]).merge()
nodes = DF(combined_regions.overlap(block, invert=True))

In [ ]:
# Sort regions and assign stable node IDs.
nodes["chrom"] = pd.Categorical(nodes["chrom"], categories=CHROMS, ordered=True)
nodes = nodes.sort_values(["chrom", "start"], ignore_index=True)
nodes["chrom"] = nodes["chrom"].astype(str)
nodes["node_id"] = nodes.index
nodes["center"] = (nodes["start"] + nodes["end"]) // 2

# Mark nodes that overlap a promoter window.
node_regions = PR(nodes[["chrom", "start", "end", "node_id"]])
promoter_nodes = DF(node_regions.overlap(promoter_regions))
nodes["is_promoter"] = nodes["node_id"].isin(promoter_nodes["node_id"])

In [ ]:
# Count ATAC reads in each node across both BAM files.
read_counts = np.zeros(len(nodes), dtype=np.int64)
total_reads = 0

for bam_path in BAMS:
    with pysam.AlignmentFile(bam_path) as bam:
        total_reads += sum(
            stats.mapped
            for stats in bam.get_index_statistics()
            if stats.contig in CHROMS
        )
        for i, (chrom, start, end) in enumerate(
            zip(nodes["chrom"], nodes["start"], nodes["end"])
        ):
            read_counts[i] += bam.count(chrom, start, end)

# Log-transformed reads per million mapped reads.
nodes["log_atac"] = np.log1p(read_counts / total_reads * 1e6)

In [ ]:
# Add the hg38 DNA sequence for each node.
GENOME_FILE = raw_path(RAW, "genome_fasta")
genome = pyfaidx.Fasta(
    str(GENOME_FILE),
    as_raw=True,
    sequence_always_upper=True,
)
nodes["sequence"] = [
    genome[chrom][start:end]
    for chrom, start, end in zip(nodes["chrom"], nodes["start"], nodes["end"])
]
print(f"{len(nodes):,} nodes, {nodes['is_promoter'].sum():,} promoters")

### Map benchmark pairs to nodes
Each tested region maps to the node with the largest overlap. Each gene maps to the node containing its TSS. Missing mappings use `-1`.

In [ ]:
node_regions = PR(nodes[["chrom", "start", "end", "node_id"]])


def map_to_nodes(regions, *, largest_overlap=False):
    """Return one node ID per region, or -1 when no node overlaps."""
    hits = DF(PR(regions.reset_index(names="pair_id")).join(
        node_regions, report_overlap=largest_overlap
    ))
    if largest_overlap:
        hits = hits.sort_values("Overlap").drop_duplicates("pair_id", keep="last")
    node_by_pair = hits.set_index("pair_id")["node_id"]
    return node_by_pair.reindex(regions.index).fillna(-1).astype(int)


# Tested element: choose the node with the most overlap.
crispr["elem_node"] = map_to_nodes(
    crispr[["chrom", "start", "end"]], largest_overlap=True
)

# Target gene: choose the node containing its one-base TSS interval.
gene_tss = crispr[["chrom", "tss"]].rename(columns={"tss": "start"})
gene_tss["end"] = gene_tss["start"] + 1
crispr["gene_node"] = map_to_nodes(gene_tss)

coverage = crispr.assign(
    elem_in_graph=crispr["elem_node"] >= 0,
    gene_in_graph=crispr["gene_node"] >= 0,
)
coverage.groupby("label")[["elem_in_graph", "gene_in_graph"]].mean()

In [ ]:
crispr.to_parquet(PROC / "crispr.parquet")
nodes.to_parquet(PROC / "nodes.parquet")

### Hi-C
Streamed from ENCODE (no 34 GB download). Keeps SCALE-normalized observed and O/E between bins that contain a node or benchmark element/TSS, within 5 Mb.

In [ ]:
# Keep only bins that contain a node, a tested element, or a gene TSS.
positions = pd.concat(
    [
        nodes[["chrom", "center"]].rename(columns={"center": "position"}),
        crispr[["chrom", "center"]].rename(columns={"center": "position"}),
        crispr[["chrom", "tss"]].rename(columns={"tss": "position"}),
    ],
    ignore_index=True,
)
positions["bin"] = positions["position"] // BIN_SIZE * BIN_SIZE
bins = positions.groupby("chrom")["bin"].unique()

hic_cache = PROC / "hic"
hic_cache.mkdir(parents=True, exist_ok=True)
chrom_lengths = {
    chromosome.name: chromosome.length
    for chromosome in hicstraw.HiCFile(HIC_URL).getChromosomes()
}

In [ ]:
def contact_block(matrix, chrom, block_start):
    """Read nearby contacts from one row block of a chromosome."""
    chrom_length = chrom_lengths[chrom]
    row_end = min(block_start + MAX_DIST, chrom_length)
    column_end = min(block_start + 2 * MAX_DIST, chrom_length)
    records = matrix.getRecords(
        block_start, row_end - 1, block_start, column_end - 1
    )
    values = np.array(
        [(record.binX, record.binY, record.counts) for record in records],
        dtype=float,
    ).reshape(-1, 3)

    bin1 = np.minimum(values[:, 0], values[:, 1]).astype(int)
    bin2 = np.maximum(values[:, 0], values[:, 1]).astype(int)
    keep = (
        (bin1 >= block_start)
        & (bin1 < block_start + MAX_DIST)
        & (bin2 - bin1 <= MAX_DIST)
        & np.isin(bin1, bins[chrom])
        & np.isin(bin2, bins[chrom])
    )
    return pd.DataFrame({
        "chrom": chrom,
        "bin1": bin1[keep],
        "bin2": bin2[keep],
        "observed": values[keep, 2],
    })


def fetch_hic(chrom):
    """Fetch one chromosome's contacts, or load its cached table."""
    cache_file = hic_cache / f"{chrom}.parquet"
    if cache_file.exists():
        return pd.read_parquet(cache_file)

    hic_file = hicstraw.HiCFile(HIC_URL)
    observed = hic_file.getMatrixZoomData(
        chrom, chrom, "observed", "SCALE", "BP", BIN_SIZE
    )
    observed_over_expected = hic_file.getMatrixZoomData(
        chrom, chrom, "oe", "SCALE", "BP", BIN_SIZE
    )
    # Initialize the remote O/E matrix before requesting its expected values.
    observed_over_expected.getRecords(0, BIN_SIZE, 0, BIN_SIZE)
    expected = np.asarray(observed_over_expected.getExpectedValues())
    if expected.size == 0:
        raise RuntimeError(f"{chrom}: incomplete remote read")

    blocks = [
        contact_block(observed, chrom, block_start)
        for block_start in range(0, chrom_lengths[chrom], MAX_DIST)
    ]
    contacts = pd.concat(blocks, ignore_index=True)
    contacts = contacts.drop_duplicates(["bin1", "bin2"])
    distance_in_bins = (contacts["bin2"] - contacts["bin1"]) // BIN_SIZE
    contacts["oe"] = contacts["observed"] / expected[distance_in_bins]
    contacts = contacts.dropna()
    contacts.to_parquet(cache_file, index=False)
    return contacts

In [ ]:
def fetch_hic_retry(chrom, tries=3):
    """Retry transient remote failures and return a small worker result."""
    for attempt in range(1, tries + 1):
        try:
            fetch_hic(chrom)
            return chrom
        except Exception as error:
            print(f"{chrom} attempt {attempt} failed: {error}")
    raise RuntimeError(f"{chrom}: failed after {tries} attempts")


# A worker may crash on a network error; restart with unfinished chromosomes.
for restart in range(5):
    pending = sorted(
        [chrom for chrom in CHROMS if not (hic_cache / f"{chrom}.parquet").exists()],
        key=chrom_lengths.get,
        reverse=True,
    )
    if not pending:
        break
    try:
        with ProcessPoolExecutor(
            max_workers=4, mp_context=mp.get_context("fork")
        ) as executor:
            list(executor.map(fetch_hic_retry, pending))
    except BrokenProcessPool:
        print("Hi-C worker crashed; restarting")

missing = [chrom for chrom in CHROMS if not (hic_cache / f"{chrom}.parquet").exists()]
if missing:
    raise RuntimeError(f"Hi-C contacts missing for: {', '.join(missing)}")

In [ ]:
hic = pd.read_parquet(hic_cache)
print(f"{len(hic):,} contacts")

# Low contacts per selected bin may indicate an incomplete chromosome read.
contacts_per_bin = hic.groupby("chrom").size().div(bins.map(len))
contacts_per_bin.describe().round(1)

### Summary statistics: contact distances
Hi-C files store only nonzero pairs. Past ~500 kb a 5 kb bin pair expects <1 read, so stored pairs have inflated O/E (single-read noise): Hi-C edges will need a minimum observed count, not just O/E ≥ τ.

In [ ]:
from scipy.signal import fftconvolve

# Compare the genomic distances of benchmark pairs and Hi-C contacts.
crispr["dist_kb"] = crispr["distanceToTSS"].abs() / 1_000
hic["dist_kb"] = (hic["bin2"] - hic["bin1"]) / 1_000

pair_distances = crispr.groupby(["Dataset", "label"])["dist_kb"].describe(
    percentiles=[0.1, 0.5, 0.9]
)
contact_distances = hic.groupby(
    pd.cut(
        hic["dist_kb"],
        [0, 10, 50, 100, 500, 1_000, 5_000],
        include_lowest=True,
    ),
    observed=True,
)[["observed", "oe"]].describe()
display(pair_distances.round(1))
display(contact_distances.round(2))


def possible_bin_pairs(bin_positions):
    """Count selected bin pairs at each separation, including zero contacts."""
    occupied = np.bincount(bin_positions // BIN_SIZE).astype(float)
    correlation = fftconvolve(occupied, occupied[::-1])
    nonnegative_distances = correlation[len(occupied) - 1:]
    return np.rint(nonnegative_distances[:MAX_DIST // BIN_SIZE + 1])


n_pairs = sum(possible_bin_pairs(chrom_bins) for chrom_bins in bins)
contact_totals = hic.groupby(
    (hic["bin2"] - hic["bin1"]) // BIN_SIZE
)["observed"].sum()
decay = contact_totals / n_pairs[contact_totals.index]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for label, color in [(False, "#2a78d6"), (True, "#eb6834")]:
    distances = crispr.loc[crispr["label"] == label, "dist_kb"]
    axes[0].hist(
        np.log10(distances),
        bins=40,
        density=True,
        histtype="step",
        lw=2,
        color=color,
        label=f"label={label}",
    )
axes[0].set(xlabel="log10 enhancer–TSS distance (kb)", ylabel="density")
axes[0].legend()

axes[1].loglog(
    decay.index[1:] * BIN_SIZE / 1_000,
    decay.values[1:],
    color="#2a78d6",
)
axes[1].set(
    xlabel="Hi-C bin distance (kb)",
    ylabel="mean observed contact (incl. zeros)",
)
plt.tight_layout()

## 2. Graph and random walks
Undirected graph over `nodes`: chain edges join consecutive nodes on a chromosome; Hi-C edges join nodes whose bins have $O/E \ge \tau$ and observed $\ge$ `MIN_OBS` (removes single-read noise, see 1).

In [ ]:
# Load the tables created in the data section.
nodes = pd.read_parquet(PROC / "nodes.parquet")
crispr = pd.read_parquet(PROC / "crispr.parquet")
hic = pd.read_parquet(PROC / "hic")

TAU = 3.0       # Minimum Hi-C observed/expected ratio.
MIN_OBS = 5.0   # Minimum observed Hi-C contact count.
OE_CAP = 100.0  # Cap large O/E values when weighting walks.

node_bins = nodes[["chrom", "node_id"]].copy()
node_bins["bin"] = nodes["center"] // BIN_SIZE * BIN_SIZE

# Chain edges join consecutive nodes on the same chromosome.
same_chromosome = nodes["chrom"].to_numpy()[1:] == nodes["chrom"].to_numpy()[:-1]
chain_edges = pd.DataFrame({
    "u": nodes["node_id"].to_numpy()[:-1][same_chromosome],
    "v": nodes["node_id"].to_numpy()[1:][same_chromosome],
    "type": 0,
    "oe": np.nan,
})

# Hi-C edges join nodes whose bins have sufficient contact evidence.
strong_contacts = hic[
    (hic["bin2"] > hic["bin1"])
    & (hic["oe"] >= TAU)
    & (hic["observed"] >= MIN_OBS)
]
left_nodes = node_bins.rename(columns={"bin": "bin1", "node_id": "u"})
right_nodes = node_bins.rename(columns={"bin": "bin2", "node_id": "v"})
hic_edges = strong_contacts.merge(left_nodes).merge(right_nodes)
hic_edges = hic_edges[hic_edges["v"] - hic_edges["u"] > 1]

edges = pd.concat([
    chain_edges,
    pd.DataFrame({
        "u": hic_edges["u"],
        "v": hic_edges["v"],
        "type": 1,
        "oe": hic_edges["oe"].clip(upper=OE_CAP),
    }),
], ignore_index=True)
centers = nodes["center"].to_numpy()
edges["dist"] = centers[edges["v"]] - centers[edges["u"]]
edges.to_parquet(PROC / "edges.parquet")

edge_summary = edges.groupby("type").agg(
    edges=("u", "size"),
    median_dist_kb=("dist", lambda distance: distance.median() / 1_000),
    median_oe=("oe", "median"),
)
edge_summary

**Walks.** node2vec second-order walk (proposal eqs. 3–5): edge weight $\pi_{uv}$ is $\beta$ for chain edges and $O/E$ for Hi-C edges, times $1/p$ to return to the previous node, $1$ to a neighbor of it, $1/q$ otherwise. `random_walk` returns the visited nodes and the CSR index of each edge taken, which carries the step's type, signed distance, and $O/E$ for tokenization.

In [ ]:
class Graph:
    """Undirected graph stored as sorted CSR adjacency arrays."""

    def __init__(self, edges, nodes):
        # Store both directions of each edge.
        source = np.r_[edges["u"], edges["v"]]
        target = np.r_[edges["v"], edges["u"]]
        order = np.lexsort((target, source))

        degree = np.bincount(source, minlength=len(nodes))
        self.indptr = np.r_[0, np.cumsum(degree)]
        self.nbr = target[order]
        self.type = np.r_[edges["type"], edges["type"]][order]
        self.oe = np.r_[edges["oe"], edges["oe"]][order]
        self.center = nodes["center"].to_numpy()
        self.is_promoter = nodes["is_promoter"].to_numpy()

    def steps(self, walk, edge_indices):
        """Show the node and incoming edge at each step of a walk."""
        return pd.DataFrame({
            "node": walk,
            "type": np.r_[-1, self.type[edge_indices]],
            "oe": np.r_[np.nan, self.oe[edge_indices]],
            "dist": np.r_[0, np.diff(self.center[walk])],
            "is_promoter": self.is_promoter[walk],
        })


def random_walk(graph, start, L=64, p=4.0, q=1.0, beta=5.0, rng=None):
    """Sample a node2vec-style walk and its traversed edge indices."""
    if rng is None:
        rng = np.random.default_rng()

    walk = [start]
    edge_indices = []
    for step in range(L):
        current = walk[-1]
        lo, hi = graph.indptr[current], graph.indptr[current + 1]
        neighbors = graph.nbr[lo:hi]
        weights = np.where(graph.type[lo:hi] == 0, beta, graph.oe[lo:hi])

        if step > 0:
            previous = walk[-2]
            previous_neighbors = graph.nbr[
                graph.indptr[previous]:graph.indptr[previous + 1]
            ]
            bias = np.where(
                neighbors == previous,
                1 / p,
                np.where(np.isin(neighbors, previous_neighbors), 1.0, 1 / q),
            )
            weights = weights * bias

        cumulative_weights = np.cumsum(weights)
        choice = lo + np.searchsorted(
            cumulative_weights,
            rng.random() * cumulative_weights[-1],
            side="right",
        )
        walk.append(graph.nbr[choice])
        edge_indices.append(choice)

    return np.array(walk), np.array(edge_indices)


g = Graph(edges, nodes)
anchor = crispr.loc[crispr["label"], "gene_node"].iloc[0]
walk, edge_indices = random_walk(g, anchor, rng=np.random.default_rng(0))
g.steps(walk, edge_indices).head(10)

In [ ]:
# Compare walk settings on 200 randomly chosen anchor nodes.
rng = np.random.default_rng(0)
diagnostic_anchors = np.random.default_rng(1).choice(
    len(nodes), 200, replace=False
)
settings = [
    (1, 1, 5),
    (4, 1, 5),
    (4, 0.5, 5),
    (4, 2, 5),
    (4, 1, 1),
    (4, 1, 20),
]

walk_diagnostics = []
for p, q, beta in settings:
    start_time = time.time()
    samples = [
        random_walk(g, anchor, p=p, q=q, beta=beta, rng=rng)
        for anchor in diagnostic_anchors
    ]
    distinct_nodes = [len(set(walk)) for walk, _ in samples]
    hic_fractions = [np.mean(g.type[edges] == 1) for _, edges in samples]
    max_reach_kb = [
        np.abs(g.center[walk] - g.center[walk[0]]).max() / 1_000
        for walk, _ in samples
    ]
    walk_diagnostics.append({
        "p": p,
        "q": q,
        "beta": beta,
        "distinct nodes": np.mean(distinct_nodes),
        "Hi-C steps": np.mean(hic_fractions),
        "median reach kb": np.median(max_reach_kb),
        "ms / walk": (time.time() - start_time) / len(diagnostic_anchors) * 1_000,
    })

pd.DataFrame(walk_diagnostics).round(2)

**All walks.** $N$ walks per benchmark node, resampled $R$ times (cycled across epochs, as in the proposal); sampled in parallel processes (before torch is imported: forking after torch/CUDA init is unsafe) and cached per walk setting.

In [ ]:
WALK = dict(L=64, p=4.0, q=1.0, beta=5.0)
N_WALKS = 8
N_RESAMPLE = 10


def _walk_chunk(task):
    chunk_anchors, seed = task
    rng = np.random.default_rng(seed)
    return [
        [random_walk(g, anchor, rng=rng, **WALK) for _ in range(N_WALKS)]
        for anchor in chunk_anchors
    ]


def sample_walks(anchors, seed=0, workers=os.cpu_count()):
    """Return node and edge walks with shape (resamples, anchors, walks, steps)."""
    repeated_anchors = np.tile(anchors, N_RESAMPLE)
    chunks = np.array_split(repeated_anchors, workers * 4)
    seeds = np.random.SeedSequence(seed).spawn(len(chunks))

    with ProcessPoolExecutor(
        max_workers=workers, mp_context=mp.get_context("fork")
    ) as executor:
        chunk_results = executor.map(_walk_chunk, zip(chunks, seeds))
        results = [walks for chunk in chunk_results for walks in chunk]

    shape = (N_RESAMPLE, len(anchors), N_WALKS, -1)
    node_walks = np.stack([
        [walk for walk, _ in anchor_walks] for anchor_walks in results
    ]).astype(np.int32)
    edge_walks = np.stack([
        [edge_indices for _, edge_indices in anchor_walks]
        for anchor_walks in results
    ]).astype(np.int32)
    return node_walks.reshape(shape), edge_walks.reshape(shape)


# Only benchmark pairs whose element and gene TSS both map to graph nodes.
pairs = crispr[
    (crispr["elem_node"] >= 0) & (crispr["gene_node"] >= 0)
].reset_index(names="row")
anchors = np.unique(np.r_[pairs["elem_node"], pairs["gene_node"]])

walk_name = (
    f"walks_L{WALK['L']}_p{WALK['p']}_q{WALK['q']}_b{WALK['beta']}"
    f"_N{N_WALKS}_R{N_RESAMPLE}.npz"
)
walk_path = PROC / walk_name
if not walk_path.exists():
    start_time = time.time()
    np.savez(walk_path, *sample_walks(anchors))
    print(f"sampled in {time.time() - start_time:.0f}s")

with np.load(walk_path) as saved_walks:
    W_np = saved_walks["arr_0"]
    E_np = saved_walks["arr_1"]
print(f"{len(anchors):,} anchors, walks {W_np.shape}")

## 3. Walk encoders: bidirectional Mamba vs. GNN
**Sequence embeddings** $s_v$: mean-pooled HyenaDNA over each node's DNA (center-cropped to 1 kb), written back to `nodes.parquet` as `seq_emb`. Needs a GPU for speed; runs once.

In [ ]:
import torch

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HYENA = "LongSafari/hyenadna-tiny-1k-seqlen-hf"


@torch.no_grad()
def embed_sequences(sequences, name=HYENA, max_len=1024, batch_size=512):
    from transformers import AutoModel

    model = AutoModel.from_pretrained(name, trust_remote_code=True)
    model = model.to(DEV).eval()

    # HyenaDNA token IDs: 4 is padding and 6 is unknown.
    lookup = np.full(256, 6)
    lookup[[ord(base) for base in "ACGTN"]] = [7, 8, 9, 10, 11]

    order = np.argsort([len(sequence) for sequence in sequences])
    embeddings = []
    for start in range(0, len(sequences), batch_size):
        batch_indices = order[start:start + batch_size]
        batch = []
        for index in batch_indices:
            sequence = sequences[index]
            crop_start = max(0, (len(sequence) - max_len) // 2)
            batch.append(sequence[crop_start:][:max_len])

        token_ids = np.full((len(batch), max(map(len, batch))), 4)
        for row, sequence in enumerate(batch):
            encoded = np.frombuffer(sequence.encode(), dtype=np.uint8)
            token_ids[row, :len(sequence)] = lookup[encoded]

        token_ids = torch.from_numpy(token_ids).to(DEV)
        mask = (token_ids != 4).unsqueeze(-1)
        hidden = model(token_ids).last_hidden_state
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        embeddings.append(pooled.float().cpu().numpy())

    return np.concatenate(embeddings)[np.argsort(order)]


if "seq_emb" not in nodes:
    nodes["seq_emb"] = list(embed_sequences(nodes["sequence"].tolist()))
    nodes.to_parquet(PROC / "nodes.parquet")

**Encoders.** Both read the same walks and the same tokens (proposal eqs. 6–7): $W_x x_v$ with $x_v = [a_v; s_v]$, plus edge-type embedding, $\rho$(signed log distance), $\rho'$(log $O/E$, Hi-C steps only).
- `mamba`: $M$ bidirectional Mamba blocks over each walk; $z_u$ = anchor token averaged over the $N$ walks (eqs. 8–9). Uses `mamba_ssm` CUDA kernels when installed, else a plain-PyTorch selective scan.
- `gnn`: $M$ GINE layers on the subgraph traversed by the anchor's $N$ walks; $z_u$ = anchor node. Same neighborhood, permutation-invariant aggregator (README: architecture swap).

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    from mamba_ssm import Mamba
except ImportError:
    Mamba = None


class TorchMamba(nn.Module):
    """Plain-PyTorch selective SSM with the mamba_ssm.Mamba interface."""

    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_in = expand * d_model
        self.r = math.ceil(d_model / 16)

        self.in_proj = nn.Linear(d_model, 2 * d_in, bias=False)
        self.out_proj = nn.Linear(d_in, d_model, bias=False)
        self.conv = nn.Conv1d(
            d_in, d_in, d_conv, groups=d_in, padding=d_conv - 1
        )
        self.x_proj = nn.Linear(d_in, self.r + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.r, d_in)
        self.A_log = nn.Parameter(
            torch.log(torch.arange(1, d_state + 1.0)).repeat(d_in, 1)
        )
        self.D = nn.Parameter(torch.ones(d_in))

    def forward(self, x):
        length = x.shape[1]
        state_size = self.A_log.shape[1]
        x, gate = self.in_proj(x).chunk(2, dim=-1)
        x = self.conv(x.transpose(1, 2))[..., :length].transpose(1, 2)
        x = F.silu(x)

        dt, B, C = self.x_proj(x).split(
            [self.r, state_size, state_size], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt)).unsqueeze(-1)
        decay = torch.exp(-dt * torch.exp(self.A_log))
        input_state = dt * B.unsqueeze(2) * x.unsqueeze(-1)

        state = torch.zeros_like(decay[:, 0])
        outputs = []
        for step in range(length):
            state = decay[:, step] * state + input_state[:, step]
            outputs.append((state * C[:, step].unsqueeze(1)).sum(-1))

        sequence = torch.stack(outputs, dim=1)
        return self.out_proj((sequence + x * self.D) * F.silu(gate))


class BiMamba(nn.Module):
    """Forward and reverse Mamba blocks with a pre-norm residual."""

    def __init__(self, d):
        super().__init__()
        if Mamba is None:
            self.fwd = TorchMamba(d)
            self.bwd = TorchMamba(d)
        else:
            self.fwd = Mamba(d_model=d)
            self.bwd = Mamba(d_model=d)
        self.norm = nn.LayerNorm(d)

    def forward(self, x):
        normalized = self.norm(x)
        return x + self.fwd(normalized) + self.bwd(normalized.flip(1)).flip(1)


class Fourier(nn.Module):
    """Map a scalar to a learned Fourier feature vector."""

    def __init__(self, d, n_freq=16, max_period=100.0):
        super().__init__()
        periods = max_period ** torch.linspace(0, 1, n_freq)
        self.register_buffer("freq", 2 * math.pi / periods)
        self.lin = nn.Linear(2 * n_freq, d)

    def forward(self, x):
        phase = x.unsqueeze(-1) * self.freq
        waves = torch.cat([phase.sin(), phase.cos()], dim=-1)
        return self.lin(waves)

In [ ]:
# Pair endpoints index into the cached anchor walks.
WALKS = torch.from_numpy(W_np).long().to(DEV)
EIDX = torch.from_numpy(E_np).long().to(DEV)
PE = torch.tensor(np.searchsorted(anchors, pairs["elem_node"]), device=DEV)
PG = torch.tensor(np.searchsorted(anchors, pairs["gene_node"]), device=DEV)

node_features = [nodes[["log_atac"]].to_numpy()]
if "seq_emb" in nodes:
    node_features.append(np.stack(nodes["seq_emb"].values))
X = torch.tensor(np.hstack(node_features), dtype=torch.float32, device=DEV)

CENTER = torch.tensor(nodes["center"].values, device=DEV)
ETYPE = torch.tensor(g.type, device=DEV)
LOGOE = torch.tensor(
    np.nan_to_num(np.log(g.oe)), dtype=torch.float32, device=DEV
)


def signed_log_distance(distance):
    distance = distance.float()
    return distance.sign() * torch.log1p(distance.abs() / 1_000)

In [ ]:
class WalkEncoder(nn.Module):
    """Encode sampled walks around one or more anchor nodes."""

    def __init__(self, kind, d=128, n_layers=2):
        super().__init__()
        self.kind = kind
        self.node = nn.Sequential(nn.LayerNorm(X.shape[1]), nn.Linear(X.shape[1], d))
        self.etype = nn.Embedding(3, d)  # chain, Hi-C, anchor
        self.rho = Fourier(d)
        self.rho_oe = Fourier(d)

        if kind == "mamba":
            self.layers = nn.ModuleList(BiMamba(d) for _ in range(n_layers))
        else:
            self.layers = nn.ModuleList(
                GINEConv(nn.Sequential(
                    nn.Linear(d, d), nn.GELU(), nn.Linear(d, d)
                ))
                for _ in range(n_layers)
            )
            self.norms = nn.ModuleList(nn.LayerNorm(d) for _ in range(n_layers))
        self.out = nn.LayerNorm(d)

    def edge_emb(self, edge_type, distance, log_oe):
        contact = self.rho_oe(log_oe) * (edge_type == 1).unsqueeze(-1)
        return self.etype(edge_type) + self.rho(distance) + contact

    def forward(self, W, E):
        if self.kind == "mamba":
            return self._encode_mamba(W, E)
        return self._encode_gnn(W, E)

    def _encode_mamba(self, W, E):
        n_anchors, n_walks, walk_length = W.shape

        def prepend(values, first):
            initial = torch.full_like(values[..., :1], first)
            return torch.cat([initial, values], dim=-1)

        edge_type = prepend(ETYPE[E], 2)
        distance = prepend(
            signed_log_distance(CENTER[W[..., 1:]] - CENTER[W[..., :-1]]),
            0,
        )
        log_oe = prepend(LOGOE[E], 0)
        tokens = self.node(X[W]) + self.edge_emb(edge_type, distance, log_oe)

        hidden = tokens.view(n_anchors * n_walks, walk_length, -1)
        for layer in self.layers:
            hidden = layer(hidden)
        anchor_hidden = self.out(hidden[:, 0])
        return anchor_hidden.view(n_anchors, n_walks, -1).mean(1)

    def _encode_gnn(self, W, E):
        n_anchors = W.shape[0]
        # Offset node IDs so each anchor's sampled subgraph stays separate.
        unique_nodes, inverse = torch.unique(
            torch.arange(n_anchors, device=W.device).view(n_anchors, 1, 1)
            * len(X) + W,
            return_inverse=True,
        )
        global_nodes = unique_nodes % len(X)
        starts = inverse[..., :-1].reshape(-1)
        ends = inverse[..., 1:].reshape(-1)

        # Deduplicate undirected walk edges, then add both directions.
        edge_keys = torch.stack([
            torch.minimum(starts, ends),
            torch.maximum(starts, ends),
            E.reshape(-1),
        ])
        left, right, edge_ids = torch.unique(edge_keys, dim=1)
        source = torch.cat([left, right])
        target = torch.cat([right, left])
        edge_ids = torch.cat([edge_ids, edge_ids])

        edge_features = self.edge_emb(
            ETYPE[edge_ids],
            signed_log_distance(
                CENTER[global_nodes[target]] - CENTER[global_nodes[source]]
            ),
            LOGOE[edge_ids],
        )
        anchor_indices = inverse[:, 0, 0]
        hidden = self.node(X[global_nodes]).index_add(
            0, anchor_indices, self.etype.weight[2].expand(n_anchors, -1)
        )
        edge_index = torch.stack([source, target])
        for conv, norm in zip(self.layers, self.norms):
            hidden = hidden + conv(norm(hidden), edge_index, edge_features)
        return self.out(hidden[anchor_indices])


class PairModel(nn.Module):
    """Predict a CRISPR label from element and gene-anchor embeddings."""

    def __init__(self, kind, d, n_layers):
        super().__init__()
        self.enc = WalkEncoder(kind, d, n_layers)
        self.mlp = nn.Sequential(
            nn.Linear(3 * d, d),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d, 1),
        )

    def forward(self, W, E, element_ids, gene_ids):
        pair_anchors, inverse = torch.unique(
            torch.cat([element_ids, gene_ids]), return_inverse=True
        )
        embeddings = self.enc(W[pair_anchors], E[pair_anchors])
        n_pairs = len(element_ids)
        element_emb = embeddings[inverse[:n_pairs]]
        gene_emb = embeddings[inverse[n_pairs:]]
        pair_features = torch.cat([
            element_emb, gene_emb, element_emb * gene_emb
        ], dim=-1)
        return self.mlp(pair_features).squeeze(-1)


print(
    "mamba backend:", "mamba_ssm" if Mamba is not None else "TorchMamba",
    "| device:", DEV,
)

**Training and evaluation.** Chromosome-held-out CV (folds balanced by pair count), class-weighted BCE (eq. 11), walks resampled each epoch, test logits averaged over 4 walk resamples. Pairs whose element is not in the graph score lowest, as in ENCODE-rE2G. Metrics: AUPRC and precision at 70% recall, per screen and pooled.

In [ ]:
CFG = {
    "d": 128,
    "layers": 2,
    "epochs": 30,
    "batch": 128,
    "lr": 1e-3,
    "wd": 1e-2,
    "folds": 5,
    "seed": 0,
}
MODELS = ["mamba", "gnn"]

# Assign whole chromosomes to folds, balancing the number of CRISPR pairs.
fold_load = np.zeros(CFG["folds"])
fold_of = {}
for chrom, n_pairs in crispr["chrom"].value_counts().items():
    fold = int(fold_load.argmin())
    fold_of[chrom] = fold
    fold_load[fold] += n_pairs
pairs["fold"] = pairs["chrom"].map(fold_of)
Y = torch.tensor(pairs["label"].values, dtype=torch.float32, device=DEV)


def fit_predict(kind, train, test):
    torch.manual_seed(CFG["seed"])
    model = PairModel(kind, CFG["d"], CFG["layers"]).to(DEV)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"]
    )
    positive_weight = (1 - Y[train]).sum() / Y[train].sum()

    for epoch in range(CFG["epochs"]):
        model.train()
        resample = epoch % N_RESAMPLE
        order = torch.randperm(len(train), device=DEV)
        for positions in order.split(CFG["batch"]):
            batch = train[positions]
            scores = model(WALKS[resample], EIDX[resample], PE[batch], PG[batch])
            loss = F.binary_cross_entropy_with_logits(
                scores, Y[batch], pos_weight=positive_weight
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        predictions = []
        for resample in range(min(4, N_RESAMPLE)):
            batch_scores = [
                model(WALKS[resample], EIDX[resample], PE[batch], PG[batch])
                for batch in test.split(512)
            ]
            predictions.append(torch.cat(batch_scores))
        return torch.stack(predictions).mean(0).cpu().numpy()


for kind in MODELS:
    for fold in range(CFG["folds"]):
        train_rows = np.where(pairs["fold"] != fold)[0]
        test_rows = np.where(pairs["fold"] == fold)[0]
        train = torch.tensor(train_rows, device=DEV)
        test = torch.tensor(test_rows, device=DEV)

        start_time = time.time()
        pairs.loc[test_rows, kind] = fit_predict(kind, train, test)
        print(f"{kind} fold {fold}: {time.time() - start_time:.0f}s")

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve


def precision_at_recall(labels, scores, target_recall=0.7):
    precision, recall, _ = precision_recall_curve(labels, scores)
    return precision[recall >= target_recall].max()


# Pairs absent from the graph receive the lowest model score.
predictions = crispr.assign(distance=-crispr["distanceToTSS"].abs())
for model_name in MODELS:
    scored_pairs = pd.Series(pairs[model_name].values, index=pairs["row"])
    predictions[model_name] = scored_pairs.reindex(predictions.index).fillna(-1e9)
predictions.to_parquet(PROC / "predictions.parquet")

datasets = list(predictions.groupby("Dataset")) + [("Pooled", predictions)]
result_rows = []
for dataset_name, dataset in datasets:
    for model_name in ["distance", *MODELS]:
        result_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "AUPRC": average_precision_score(
                dataset["label"], dataset[model_name]
            ),
            "P@70%R": precision_at_recall(
                dataset["label"], dataset[model_name]
            ),
        })

results = pd.DataFrame(result_rows)
results.to_csv(PROC / "results.csv", index=False)
results.pivot(index="model", columns="dataset").round(3)